## Загрузка данных — запустите как есть

In [2]:
# Загрузка: в Colab нажмите «Выбрать файлы» и укажите orders.csv из LMS.
import os
if not os.path.exists("orders.csv"):
    try:
        from google.colab import files
        files.upload()
    except ImportError:
        print("Положите orders.csv в одну папку с ноутбуком")
FILE = "orders.csv"
print("Работаем с файлом:", FILE)

Saving orders.csv to orders.csv
Работаем с файлом: orders.csv


In [3]:
import csv
import json
import numpy as np

with open(FILE, encoding="utf-8") as f:
    rows = list(csv.DictReader(f))

price, qty, city, category, status, month, product = [], [], [], [], [], [], []
for r in rows:
    price.append(int(r["price"]))
    qty.append(int(r["qty"]))
    city.append(r["city"])
    category.append(r["category"])
    status.append(r["status"])
    month.append(int(r["date"].split("-")[1]))
    product.append(r["product"])

price = np.array(price)
qty = np.array(qty)
city = np.array(city)
category = np.array(category)
status = np.array(status)
month = np.array(month)
product = np.array(product)

revenue = price * qty
paid = status == "оплачен"
print(revenue.shape, revenue[:5])

(500,) [198000  24000 159000 189000 129000]


## [База]

### Задача 1. Цены выросли на 5%
Магазин поднимает все цены на 5%. Посчитайте новые суммы заказов **одной строкой**, без цикла.
На сколько тенге вырастет общая выручка?

**Пример проверки:** выручка вырастет на `4 625 325 ₸`.

In [8]:
def tg(x):
    return f"{x:,.0f}".replace(",", " ") + " ₸"

new_revenue = price * 1.05 * qty          # новые суммы заказов одной строкой
growth = new_revenue.sum() - revenue.sum()
print(f"Выручка вырастет на {tg(growth)} — столько магазин получит сверху при том же количестве заказов.")

Выручка вырастет на 4 625 325 ₸ — столько магазин получит сверху при том же количестве заказов.


### Задача 2. Смартфоны
С помощью маски по `category` найдите:
- сколько заказов в категории «Смартфоны»;
- на какую сумму;
- какая это доля от всех заказов.

**Пример проверки:** `131` заказ на `30 689 000 ₸`, доля `26.2%`.

In [9]:
is_phone = category == "Смартфоны"
n = is_phone.sum()                         # True считается как 1
total = revenue[is_phone].sum()
share = is_phone.mean() * 100              # доля True среди всех заказов
print(f"{n} заказ на {tg(total)}, доля {share:.1f}% — примерно каждый четвёртый заказ магазина приходится на смартфоны.")

131 заказ на 30 689 000 ₸, доля 26.2% — примерно каждый четвёртый заказ магазина приходится на смартфоны.


### Задача 3. Два условия
Сколько **оплаченных** заказов из **Шымкента** и на какую сумму? Не забудьте скобки вокруг каждого условия.

**Пример проверки:** `71` заказ на `12 852 500 ₸`.

In [10]:
mask = (city == "Шымкент") & paid
print(f"{mask.sum()} заказ на {tg(revenue[mask].sum())} — столько денег из Шымкента уже реально поступило.")

71 заказ на 12 852 500 ₸ — столько денег из Шымкента уже реально поступило.


### Задача 4. Неделя курьера
Первые 30 заказов — это 6 дней по 5 заказов: `w = revenue[:30].reshape(6, 5)`.
1. Сумма за каждый день — какую `axis` взять? Сколько чисел должно получиться?
2. В какой день (номер с 1) сумма самая большая?
3. Самый дорогой заказ **каждого** дня.

**Пример проверки:** лучший — **3-й** день, `1 094 000 ₸`; самые дорогие по дням — `[198000 319000 549000 449000 449000 319000]`.

In [11]:
w = revenue[:30].reshape(6, 5)

day_sums = w.sum(axis=1)                  # 6 чисел — по одному на каждый день
best_day = day_sums.argmax() + 1          # +1, потому что дни считаем с 1
max_per_day = w.max(axis=1)               # самый дорогой заказ каждого дня

print("Суммы по дням:", day_sums)
print(f"Лучший — {best_day}-й день, {tg(day_sums.max())} — в этот день курьер привёз больше всего денег.")
print("Самые дорогие по дням:", max_per_day, "— видно, какой заказ «делал» каждый день.")

Суммы по дням: [ 699000  925000 1094000  636000 1047000  571000]
Лучший — 3-й день, 1 094 000 ₸ — в этот день курьер привёз больше всего денег.
Самые дорогие по дням: [198000 319000 549000 449000 449000 319000] — видно, какой заказ «делал» каждый день.


## [Челлендж]

### Задача 5. Заказы в трёх валютах
Покупатели из-за рубежа просят показать суммы заказов в своей валюте.
Курсы Нацбанка РК на 19 ноября 2026: 1 USD = 444.16 ₸, 1 EUR = 487.29 ₸, 1 RUB = 5.40 ₸.

1. Возьмите суммы **первых 5 заказов** из `revenue` и получите **одной операцией** таблицу «заказ × валюта» формы `(5, 3)`, округлите до 2 знаков.
   Какую форму нужно дать суммам заказов, чтобы сработал broadcasting?
2. Выведите рядом товар каждого из этих 5 заказов — массив `product` уже собран в ячейке загрузки.


**Пример проверки:** форма суммам — `(5, 1)`; первый заказ (Xiaomi Redmi Note 13, 198 000 ₸) — `[445.79  406.33  36666.67]`.

In [12]:
rates = np.array([444.16, 487.29, 5.40])          # USD, EUR, RUB — форма (3,)
amounts = revenue[:5].reshape(5, 1)               # форма (5, 1) — столбец

table = np.round(amounts / rates, 2)              # одна операция → форма (5, 3)

print("Форма сумм:", amounts.shape)
print("Товары:", product[:5])
print(table)
print(f"Первый заказ ({product[0]}, {tg(revenue[0])}) — {table[0]}: столько покупатель заплатил бы в USD, EUR и RUB.")

Форма сумм: (5, 1)
Товары: ['Xiaomi Redmi Note 13' 'Клавиатура Logitech K120' 'Samsung Galaxy Tab A9'
 'Samsung Galaxy A55' 'Монитор LG 27 дюймов']
[[  445.79   406.33 36666.67]
 [   54.03    49.25  4444.44]
 [  357.98   326.29 29444.44]
 [  425.52   387.86 35000.  ]
 [  290.44   264.73 23888.89]]
Первый заказ (Xiaomi Redmi Note 13, 198 000 ₸) — [  445.79   406.33 36666.67]: столько покупатель заплатил бы в USD, EUR и RUB.


### Задача 6. Категории
Для **каждой** категории из `np.unique(category)` выведите число заказов, выручку и средний чек (цикл по категориям можно).
Какая категория приносит больше всего денег? А у какой самый большой средний чек?

**Пример проверки:** и по выручке, и по среднему чеку лидируют **Ноутбуки** — `40 992 000 ₸`, средний чек ≈ `422 598 ₸`.

In [13]:
cats = np.unique(category)
counts, sums, avgs = [], [], []

for c in cats:
    m = category == c                      # маска для одной категории
    counts.append(m.sum())
    sums.append(revenue[m].sum())
    avgs.append(revenue[m].mean())
    print(f"{c}: {m.sum()} заказов, выручка {tg(revenue[m].sum())}, средний чек {tg(revenue[m].mean())}")

sums = np.array(sums)
avgs = np.array(avgs)

top_rev = cats[sums.argmax()]
top_avg = cats[avgs.argmax()]
print(f"Больше всего денег приносят {top_rev} — {tg(sums.max())}: это главная категория магазина по выручке.")
print(f"Самый большой средний чек у категории {top_avg} — ≈ {tg(avgs.max())}: один такой заказ стоит дороже всего.")

Аксессуары: 171 заказов, выручка 4 514 500 ₸, средний чек 26 401 ₸
Мониторы: 50 заказов, выручка 6 046 000 ₸, средний чек 120 920 ₸
Ноутбуки: 97 заказов, выручка 40 992 000 ₸, средний чек 422 598 ₸
Планшеты: 51 заказов, выручка 10 265 000 ₸, средний чек 201 275 ₸
Смартфоны: 131 заказов, выручка 30 689 000 ₸, средний чек 234 267 ₸
Больше всего денег приносят Ноутбуки — 40 992 000 ₸: это главная категория магазина по выручке.
Самый большой средний чек у категории Ноутбуки — ≈ 422 598 ₸: один такой заказ стоит дороже всего.


### Задача 7. Бесплатная доставка
Доставка бесплатна, если сумма заказа от 50 000 ₸. С помощью `np.where` подпишите каждый заказ: `"бесплатно"` или `"платно"`.
Сколько заказов получили бесплатную доставку?

**Пример проверки:** `354`.

In [14]:
delivery = np.where(revenue >= 50000, "бесплатно", "платно")
free = (delivery == "бесплатно").sum()

print(delivery[:10])
print(f"{free} заказа получили бесплатную доставку — магазин оплачивает доставку для большинства покупателей.")

['бесплатно' 'платно' 'бесплатно' 'бесплатно' 'бесплатно' 'бесплатно'
 'бесплатно' 'бесплатно' 'бесплатно' 'бесплатно']
354 заказа получили бесплатную доставку — магазин оплачивает доставку для большинства покупателей.
